In [ ]:
import os
import socket
import boto3
import pandas as pd
import pyarrow
import delta

print("Python: OK")
print("pandas:", pd.__version__)
print("pyarrow:", pyarrow.__version__)
print("boto3: OK")
print("delta-spark: OK")import socket

host = "rustfs"
port = 9000

print("RustFS IP:", socket.gethostbyname(host))

with socket.create_connection((host, port), timeout=5):
    print(f"TCP connection to {host}:{port}: OK")

In [ ]:
import os
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://rustfs:9000",
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name="us-east-1",
)

response = s3.list_buckets()

print("RustFS S3 API: OK")
print("Buckets:", [b["Name"] for b in response.get("Buckets", [])])

In [ ]:
bucket = "lakehouse"

existing = [b["Name"] for b in s3.list_buckets().get("Buckets", [])]

if bucket not in existing:
    s3.create_bucket(Bucket=bucket)
    print(f"Created bucket: {bucket}")
else:
    print(f"Bucket already exists: {bucket}")

print("Bucket test: OK")

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("LakehouseBasicTest")
    .master("spark://spark:7077")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

result = spark.range(10).count()

print("Result:", result)

assert result == 10

print("Spark Master → Worker: OK")

In [ ]:
df = spark.createDataFrame(
    [
        (1, "Siva", "India"),
        (2, "Spark", "Lakehouse"),
        (3, "RustFS", "Object Storage"),
    ],
    ["id", "name", "source"]
)

df.show()

In [ ]:
access_key = os.environ["AWS_ACCESS_KEY_ID"]
secret_key = os.environ["AWS_SECRET_ACCESS_KEY"]

spark.conf.set(
    "spark.hadoop.fs.s3a.endpoint",
    "http://rustfs:9000"
)

spark.conf.set(
    "spark.hadoop.fs.s3a.access.key",
    access_key
)

spark.conf.set(
    "spark.hadoop.fs.s3a.secret.key",
    secret_key
)

spark.conf.set(
    "spark.hadoop.fs.s3a.path.style.access",
    "true"
)

spark.conf.set(
    "spark.hadoop.fs.s3a.connection.ssl.enabled",
    "false"
)

spark.conf.set(
    "spark.hadoop.fs.s3a.impl",
    "org.apache.hadoop.fs.s3a.S3AFileSystem"
)

print("Spark S3A configuration: OK")

In [ ]:
parquet_path = "s3a://lakehouse/test/parquet"

df.write.mode("overwrite").parquet(parquet_path)

print("Parquet write: OK")
print("Path:", parquet_path)

In [ ]:
read_df = spark.read.parquet(parquet_path)

read_df.show()

count = read_df.count()

print("Rows read:", count)

assert count == 3

print("Parquet read-back: OK")

In [ ]:
response = s3.list_objects_v2(
    Bucket="lakehouse",
    Prefix="test/parquet/"
)

for obj in response.get("Contents", []):
    print(obj["Key"])

print("\nRustFS object check: OK")

In [ ]:
spark.stop()

print("Spark stopped")